# SafeCare AI
## Gemma 4-powered healthcare risk screening for rural women in India

## Problem Statement
Rural women in India often delay medical care because of social barriers, fear, stigma, and limited healthcare access.
SafeCare AI uses Gemma 4 fine-tuned with LoRA to analyze symptom descriptions in Hindi, identify severity levels, detect abuse-related risks, and provide structured healthcare guidance.

This project uses:
- Gemma 4
- LoRA fine-tuning
- 4-bit quantization
- Unsloth optimization
- Structured JSON generation
- Hindi healthcare instruction tuning

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
# This forces the notebook to only see one T4 GPU, preventing the multi-GPU freeze
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip uninstall unsloth unsloth_zoo -y
!pip install --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q datasets "trl<=0.11.0" "transformers<=4.45.2" accelerate
!rm -rf /kaggle/working/unsloth_compiled_cache
print("✅ Done — Run → Restart Session now")

In [1]:
import os

print("--- FOLDERS IN /kaggle/input/ ---")
print(os.listdir('/kaggle/input/'))
print("\n--- ALL FILES IN INPUT ---")

# This will dig through everything and print the exact, true path of every file
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if "gemma" in root.lower(): # Only print gemma related files to keep it clean
            print(os.path.join(root, file))

--- FOLDERS IN /kaggle/input/ ---
['datasets', 'models']

--- ALL FILES IN INPUT ---
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/generation_config.json


## Gemma 4 Fine-Tuning Setup

In [2]:
import glob

# Using the true path revealed by your diagnostic test
model_dir = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/*"
model_paths = glob.glob(model_dir)

for p in model_paths:
    print(p)

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/generation_config.json


In [3]:
import os
os.makedirs("/kaggle/working/safecare/data",    exist_ok=True)
os.makedirs("/kaggle/working/safecare/outputs", exist_ok=True)
print("✅ Directories ready")

✅ Directories ready


In [ ]:
!pip install --upgrade --no-deps unsloth
!pip install --upgrade transformers

In [ ]:
!pip install liger-kernel trl peft

## LoRA Configuration

In [4]:
from unsloth import FastModel
import torch

# Updated to the EXACT true path we found in the previous step
MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=512,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
)

print("✅ Model + LoRA ready!")
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Device does not support bfloat16. Will change to float16.


==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.8.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

[transformers] Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


✅ Model + LoRA ready!
trainable params: 11,829,248 || all params: 7,952,930,080 || trainable%: 0.1487


## Dataset Preparation

In [6]:
import pandas as pd, json, glob, os

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
if csv_files:
    CSV_PATH = csv_files[0]
    print(f"✅ Found: {CSV_PATH}")
else:
    raise FileNotFoundError("Upload your CSV via '+ Add Input' → Upload first!")

df = pd.read_csv(CSV_PATH)
print(f"   Loaded {len(df)} rows")

os.makedirs("/kaggle/working/safecare/data", exist_ok=True)
training_data = []

for _, row in df.iterrows():
    raw_desc = str(row.get("Raw Description (Hindi)", "")).strip()
    if not raw_desc or raw_desc.lower() in ["nan", ""]:
        continue
    severity  = str(row.get("Severity", "medium")).strip().lower()
    barriers  = str(row.get("Barriers/Context", "")).strip()
    abuse_flag = str(row.get("Abuse Flags", "no")).strip().lower()
    region    = str(row.get("Region", "rural India")).strip()
    english   = str(row.get("English Translation", "")).strip()

    abuse_risk = []
    if abuse_flag not in ["no","nan","none",""]: abuse_risk.append(abuse_flag)
    if "husband" in barriers.lower() or "control" in barriers.lower():
        abuse_risk.append("partner control")

    action_steps = {
        "high":   ["Turant sarkari aspatal jayein","Mahila Helpline 181 par call karein","Kisi vishwasniya mahila ko batayein"],
        "medium": ["Is hafte ANM se milein","Apne lakshan likhein","Helpline 181 available hai"],
        "low":    ["Lakshan badhe to doctor se milein","ANM se poochhen"]
    }.get(severity, ["ANM se milein"])

    training_data.append({
        "instruction": f"महिला ने कहा: \"{raw_desc}\"\nRegion: {region}",
        "output": json.dumps({
            "severity": severity if severity in ["high","medium","low"] else "medium",
            "symptom_analysis": english if english and english != "nan" else raw_desc[:100],
            "abuse_risk_flags": abuse_risk,
            "action_steps": action_steps,
            "safe_resources": ["Mahila Helpline: 181","NCW: 7827170170","Nearest PHC"]
        }, ensure_ascii=False)
    })

json_path = "/kaggle/working/safecare/data/safecare_training.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)
print(f"✅ {len(training_data)} training examples saved")

✅ Found: /kaggle/input/datasets/anushkapatel9359/mymodel/SafeCare_Sample.csv
   Loaded 78 rows
✅ 78 training examples saved


## Training Pipeline

In [7]:
from datasets import load_dataset

# Use the text tokenizer inside the processor, not the processor itself
text_tokenizer = tokenizer.tokenizer

dataset = load_dataset("json", data_files=json_path, split="train")
split = dataset.train_test_split(test_size=0.15, seed=42)

def format_and_tokenize(example):
    text = (
        f"<start_of_turn>user\n"
        f"You are SafeCare, a health assistant for rural women in India. "
        f"Analyze symptoms and respond ONLY with JSON.\n\n"
        f"{example['instruction']}<end_of_turn>\n"
        f"<start_of_turn>model\n{example['output']}<end_of_turn>"
    )
    tokens = text_tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tok = split["train"].map(format_and_tokenize, remove_columns=["instruction","output"])
val_tok   = split["test"].map(format_and_tokenize,  remove_columns=["instruction","output"])
print(f"✅ Train: {len(train_tok)} | Val: {len(val_tok)}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

✅ Train: 66 | Val: 12


In [8]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

print("🚀 Starting manual training...")

# Setup optimizer
optimizer = AdamW(model.parameters(), lr=2e-4)
model.train()

total_steps = 0
for epoch in range(3):
    epoch_loss = 0
    batch_count = 0
    
    # Training
    print(f"\n📚 Epoch {epoch + 1}/3 - Training")
    for batch_idx, batch in enumerate(tqdm(train_tok)):
        # Move batch to GPU
        input_ids = torch.tensor([batch['input_ids']]).to(model.device)
        attention_mask = torch.tensor([batch['attention_mask']]).to(model.device)
        labels = torch.tensor([batch['labels']]).to(model.device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        
        # Gradient accumulation (every 4 steps)
        if (batch_idx + 1) % 4 == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        epoch_loss += loss.item()
        batch_count += 1
        total_steps += 1
        
        # Log
        if (batch_idx + 1) % 5 == 0:
            avg_loss = epoch_loss / batch_count
            print(f"  Step {batch_idx + 1}, Loss: {avg_loss:.4f}")
    
    # Final optimizer step
    optimizer.step()
    optimizer.zero_grad()
    
    # Validation
    print(f"  ✓ Epoch {epoch + 1} Training Loss: {epoch_loss / batch_count:.4f}")
    
    # Eval on validation set
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_tok, desc="Evaluating"):
            input_ids = torch.tensor([batch['input_ids']]).to(model.device)
            attention_mask = torch.tensor([batch['attention_mask']]).to(model.device)
            labels = torch.tensor([batch['labels']]).to(model.device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            val_loss += outputs.loss.item()
    
    print(f"  ✓ Epoch {epoch + 1} Validation Loss: {val_loss / len(val_tok):.4f}")
    model.train()

print("\n✅ Training complete!")

🚀 Starting manual training...

📚 Epoch 1/3 - Training


  0%|          | 0/66 [00:00<?, ?it/s]

Unsloth: Will smartly offload gradients to save VRAM!


  2%|▏         | 1/66 [00:35<38:08, 35.20s/it]

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


...

  Step 65, Loss: 14.1049


100%|██████████| 66/66 [01:10<00:00,  1.06s/it]


  ✓ Epoch 3 Training Loss: 14.1091


Evaluating: 100%|██████████| 12/12 [00:04<00:00,  2.49it/s]

  ✓ Epoch 3 Validation Loss: 14.2669

✅ Training complete!


In [9]:
SAVE_PATH = "/kaggle/working/safecare/outputs/final-model"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Saved to {SAVE_PATH}")

[transformers] Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/safecare/outputs/final-model/tokenizer_config.json.


✅ Saved to /kaggle/working/safecare/outputs/final-model


## Evaluation and Inference

In [ ]:
!pip install -q gradio

In [10]:
import gradio as gr
import torch
import json

model.eval()

SYSTEM_PROMPT = """
You are SafeCare AI, a healthcare assistant for rural women in India.

You must:
- Analyze symptoms carefully
- Detect severity level
- Identify possible abuse or control indicators
- Respond ONLY in valid JSON
- Encourage professional medical help for severe symptoms
"""

def analyze(symptom):

    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"महिला ने कहा: \"{symptom}\""
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

    inputs = text_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            pad_token_id=text_tokenizer.eos_token_id
        )

    response = text_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Describe symptoms in Hindi..."
    ),
    outputs=gr.Textbox(lines=12),
    title="SafeCare AI",
    description="Gemma 4-powered healthcare risk screening assistant for rural women in India",
    examples=[
        ["Mujhe 3 mahine se bahut bleeding ho rahi hai"],
        ["Mere chest mein ganthi hai lekin pati doctor ke paas nahi jaane deta"],
        ["Bahut thakaan rehti hai aur weight kam ho raha hai"]
    ]
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://beadef3ff7643436fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
